## Task-01

In [1]:
import os
import glob
import shutil
import pandas as pd


In [2]:
# Move all CSV files to a backup folder
csv_files = glob.glob("csv_files/*.csv")
for file in csv_files:
  shutil.move(file, "backup_folder/")
  print(f"Moved file: {file}")


Moved file: csv_files/advertising.csv
Moved file: csv_files/Real estate.csv
Moved file: csv_files/50_Startups.csv
Moved file: csv_files/Breast Cancer Classification.csv


In [3]:
# Automating Export
def export_data(df, filename, format):
  if format == "csv":
    df.to_csv(filename, index=False)
    print(f"Data exported to {filename} in CSV format.")
  elif format == "json":
    df.to_json(filename, orient="records")
    print(f"Data exported to {filename} in JSON format.")
  else:
    print("Unsupported format.")

# Example usage:
# Creating a sample dataframe
data = {'Name': ['Alice', 'Bob', 'Charlie'],
'Age': [25, 30, 35],
'City': ['New York', 'Los Angeles', 'Chicago']}
df = pd.DataFrame(data)
# Exporting to CSV
export_data(df, "output.csv", "csv")
# Exporting to JSON
export_data(df, "output.json", "json")

Data exported to output.csv in CSV format.
Data exported to output.json in JSON format.


## Task-02

In [4]:
!pip install yfinance --upgrade

In [5]:

import sqlite3
import yfinance as yf
import pandas as pd
import time

# Database setup
DB_NAME = "stocks.db"  # Use constants for database name
CONN = sqlite3.connect(DB_NAME)
CURSOR = CONN.cursor()

CURSOR.execute('''
    CREATE TABLE IF NOT EXISTS stock_data (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        symbol TEXT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
        open REAL,
        high REAL,
        low REAL,
        close REAL,
        volume INTEGER
    )
''')
CONN.commit()

# Function to fetch stock data
def fetch_stock_data(symbol):
    try:
        stock = yf.Ticker(symbol)
        data = stock.history(period="1d", interval="1m")

        if data.empty:
            print(f"No data found for {symbol}. Skipping...")
            return None  # Return None if no data is available

        latest = data.iloc[-1]  # Get the most recent price data
        return {
            "symbol": symbol,
            "open": latest["Open"],
            "high": latest["High"],
            "low": latest["Low"],
            "close": latest["Close"],
            "volume": latest["Volume"]
        }
    except Exception as e:
        print(f"Error fetching data for {symbol}: {e}")
        return None

# Function to store data in SQLite
def store_data(symbol):
    stock_data = fetch_stock_data(symbol)
    if stock_data:  # Only store if data is available
        CURSOR.execute('''
            INSERT INTO stock_data (symbol, open, high, low, close, volume)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (
            stock_data["symbol"],
            stock_data["open"],
            stock_data["high"],
            stock_data["low"],
            stock_data["close"],
            stock_data["volume"]
        ))
        CONN.commit()
        print(f"Stored data for {symbol}")

# Function to analyze stock data
def analyze_stock(symbol):
    df = pd.read_sql_query(
        "SELECT * FROM stock_data WHERE symbol=? ORDER BY timestamp DESC LIMIT 100",
        CONN,
        params=(symbol,)
    )
    print(df)

# Example Usage
SYMBOL = "MSFT"  # Microsoft stock. Use constant for symbol.
FETCH_COUNT = 5 # How many times to fetch

for _ in range(FETCH_COUNT):  # Fetch data multiple times
    store_data(SYMBOL)
    time.sleep(30)  # Wait for 1 minute before fetching again

analyze_stock(SYMBOL)

# Close database connection (use try...finally)
try:
    CONN.close()
except Exception as e:
    print(f"Error closing connection: {e}")

Stored data for MSFT
Stored data for MSFT
Stored data for MSFT
Stored data for MSFT
Stored data for MSFT
   id symbol            timestamp        open        high         low  \
0   5   MSFT  2025-02-23 05:32:04  408.315002  408.380005  407.970001   
1   4   MSFT  2025-02-23 05:31:34  408.315002  408.380005  407.970001   
2   3   MSFT  2025-02-23 05:31:03  408.315002  408.380005  407.970001   
3   2   MSFT  2025-02-23 05:30:33  408.315002  408.380005  407.970001   
4   1   MSFT  2025-02-23 05:30:03  408.315002  408.380005  407.970001   

        close  volume  
0  408.140015  482718  
1  408.140015  482718  
2  408.140015  482718  
3  408.140015  482718  
4  408.140015  482718  


## Task-03.

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "http://quotes.toscrape.com/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def get_quotes(url):
      response = requests.get(url, headers=HEADERS)
      response.raise_for_status()
      soup = BeautifulSoup(response.text, "html.parser")
      quotes = soup.find_all("div", class_="quote")

      quote_list = []
      for quote in quotes:
          text = quote.find("span", class_="text").text
          author = quote.find("small", class_="author").text
          tags = [tag.text for tag in quote.find_all("a", class_="tag")]

          quote_list.append({"Quote": text, "Author": author, "Tags": tags})

      return quote_list


quotes_data = get_quotes(URL)
df = pd.DataFrame(quotes_data)
df.to_csv("quotes.csv", index=False)
print("Data saved to quotes.csv")

Data saved to quotes.csv
